# **Filtering 2D**

<div style="color:#777777;margin-top: -15px;">
<b>Author</b>: Norman Juchler |
<b>Course</b>: ADLS ISP |
<b>Version</b>: v1.2 <br><br>
<!-- v1.2, 12.04.2025: Refactored text -->
</div>

In this notebook, we will explore image filtering in both the spatial and frequency domains. We will also examine the concept of convolution and its role in image processing.

## **Exercises**
* [Exercise 1: Spectral filtering](#exercise1)  
* [Exercise 2: Interpretation of spectra](#exercise2)  
* [Exercise 3: Spatial filtering with kernels](#exercise3)  
* [Exercise 4: Edge detection](#exercise4)  
* [Exercise 5: Border handling](#exercise5)


---

## **Preparations**

The usual preparations... The package `isp` provides some helper functions to easily render images in this Jupyter notebook.

In [ ]:
import sys
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import scipy.fft
import scipy.signal

# Enable vectorized output (for nicer plots)
%config InlineBackend.figure_formats = ["svg"]

# Inline backend configuration
%matplotlib inline

# Functionality related to this course
sys.path.append("..")
import isp

# Jupyter / IPython configuration:
# Automatically reload modules when modified, if the extension is available
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass

---

<a id='exercise1'></a>

## **☆ Exercise 1: Spectral filtering**

The photo *moon-walk.jpg* shows astronaut Harrison Schmitt walking on the Moon during the Apollo 17 mission in 1972. In this exercise, the image has been artificially distorted by adding periodic noise. If you are curious about how this distortion was generated, you can find the corresponding code in the Supplemental Material section below. The goal of this exercise is to remove the distortion using **spectral filtering**.

Such regular, periodic distortions were common in the early days of digital printing, particularly with the [halftone technique](https://en.wikipedia.org/wiki/Halftone). This method simulates continuous-tone images (i.e., smooth intensity variations) by varying the size or spacing of discrete dots. As a result, visible and often repetitive patterns can appear, especially when the printing resolution is limited.


<div style="display: flex; justify-content: center; gap: 20px; text-align: center;">
  <div style="border: 1px solid #ccc; border-radius: 3px; padding: 10px; display: inline-block;">
    <p style="margin: 0 0 8px 0; font-weight: bold">Original: moon-walk.jpg</p>
    <img src="../data/images/moon-walk-square.jpg" alt="moon-walk" style="max-height: 300px; max-width: 100%;">
  </div>
  <div style="border: 1px solid #ccc; border-radius: 3px; padding: 10px; display: inline-block;">
    <p style="margin: 0 0 8px 0; font-weight: bold">moon-walk-distorted-w2.5.jpg</p>
    <img src="../data/images/moon-walk-distorted-w2.5.jpg" alt="moon-walk-distorted-w2.5" style="max-height: 300px; max-width: 100%;">
  </div>
  <div style="border: 1px solid #ccc; border-radius: 3px; padding: 10px; display: inline-block;">
    <p style="margin: 0 0 8px 0; font-weight: bold">moon-walk-distorted-w5.jpg</p>
    <img src="../data/images/moon-walk-distorted-w5.jpg" alt="moon-walk-distorted-w5" style="max-height: 300px; max-width: 100%;">
  </div>
</div>


### **Instructions**

1. Load the images *moon-walk-distorted-w5.jpg* and *moon-walk-distorted-w2.5.jpg* from the folder *../data/images/* and display them.  
2. Compute the **Fourier transform** of each image and visualize both the **magnitude** and **phase** spectra.  
3. Load different filter masks from the subfolder *../data/images/moon-walk-masks/* and explore their structure.  
4. Apply the masks in the frequency domain to suppress the frequency components responsible for the periodic noise.  
5. Compute the **inverse Fourier transform** to reconstruct the filtered images.  
6. Repeat the process using different masks and for additional distortion levels (e.g., *w = 5*, *w = 7.5*).  
7. Discuss your results and reflect on the effectiveness of the different filtering approaches.

**Note:** Using the code provided in the *Supplemental Material* section below, you can generate distorted images with varying levels of distortion. The parameter *w* defines the spatial period (i.e., wavelength in pixels) of the distortion pattern. For example, the image *moon-walk-distorted-w5.jpg* was generated using a periodic pattern with a spatial period of *w = 5*.  

In [ ]:
######################
###    EXERCISE    ###
######################

# 1) Load the image "moon-walk-distorted-w2.5.jpg" and display it.
#    (...or moon-walk-distorted-w5.jpg, or moon-walk-distorted-w7.5.jpg)
img = ...
isp.show_image(img)
                
# 2) Compute the Fourier transform and display the magnitude / phase.
ft = ...
ft_amp = ...
ft_phase = ...

isp.show_image_pair(ft_amp, ft_phase, 
                    title1="Amplitude spectrum", 
                    title2="Phase spectrum", 
                    normalize=True)

# 3) Load a filter mask of the subfolder ../data/images/moon-walk-masks/
mask = ...
isp.show_image(mask)

# 4) Apply the mask to the Fourier transform 
ft_filt = ...

# 5) Compute the inverse Fourier transform and display the result
img_filt = ...
isp.show_image(img_filt)

# 6) Repeat the process for other masks and distortion levels (w=5, w=7.5)
# 7) Discuss the results

In [ ]:
######################
###    SOLUTION    ###
######################

# Choose between 2.5, 5, and 7.5
wavelength = 5

# 1) Load the image and display it
path_image = "../data/images/moon-walk-distorted-w%g.jpg" % wavelength
image = cv.imread(path_image, cv.IMREAD_GRAYSCALE)
isp.show_image(image, title="Original image (w=%g)" % wavelength)

# 2) Compute the Fourier transform and display the magnitude / phase
ft = scipy.fft.fft2(image)
ft = scipy.fft.fftshift(ft)
# Logarithmic scale, to enhance visibility, add 1 to avoid log(0)
ft_amp = np.log(np.abs(ft)+1)
ft_phase = np.angle(ft)
isp.show_image_pair(ft_amp, ft_phase, 
                    title1="Amplitude spectrum", 
                    title2="Phase spectrum", 
                    figsize=(9, 5),
                    normalize=True)

# Repeat the exercise for the different predefined masks
label_map = {
    1: "Low-pass filter",
    2: "High-pass filter",
    3: "Selective-pass filter (step)",
    4: "Selective-pass filter (linear)",
    5: "Selective-pass filter (Gaussian)",
    6: "Band-pass filter (step)",
    7: "Band-pass filter (linear)",
    8: "Band-pass filter (Gaussian)",
}

for i in range(1, 9):
    
    mask_no = i
    
    # 3) Load the mask
    path_mask = "../data/images/moon-walk-masks/mask%d-w%g.png" % (mask_no, wavelength)
    mask = cv.imread(path_mask, cv.IMREAD_GRAYSCALE)
    
    # 4) Apply the mask to the Fourier transform
    ft_filt = ft*mask
    ft_amp_filt = ft_amp*mask
    
    # 5) Compute the inverse Fourier transform and display the result
    ft_filt = scipy.fft.ifftshift(ft_filt)
    image_rec = np.abs(scipy.fft.ifft2(ft_filt))
    image_rec = (image_rec - image_rec.min()) / (image_rec.max() - image_rec.min()) * 255
    
    # Display the result
    isp.show_header("Mask %d: %s" % (mask_no, label_map[i]))
    isp.show_image_grid([ft_amp_filt, image_rec],
                        titles=["Amplitude spectrum (after applying mask)", "Filtered image"],
                        suppress_info=True,
                        ncols=2)
    
    # cv.imwrite("moon-walk-distorted-w%g-mask%d-rec.png" % (wavelength, mask_no), image_rec) 
    
    
# 6) Repeat the process for other masks and distortion levels (w=2.5, w=7.5)
#    --> Set different values for 'wavelength' and 'mask_no'

# 7) Discuss the results
#    --> See below

### **Observations:**

* The **low-pass filter** (mask 1) suppresses high-frequency components, resulting in a smoother but noticeably blurred image. While the periodic artifacts are reduced, fine details are lost.
* The **high-pass filter** (mask 2) removes low-frequency content but retains most of the periodic artifacts. At the same time, it strongly attenuates the main image structure, making it unsuitable for this task.
* The **selective-pass** and **band-pass filters** (masks 3–8) target specific frequency components associated with the distortion pattern. These filters effectively reduce the artifacts while preserving most of the relevant image content.
* The **selective-pass filters** are typically less aggressive than band-pass filters, often resulting in a more natural visual appearance.
* As known from 1D signal processing, ideal (i.e., sharp) spectral filters can introduce ringing artifacts in the spatial domain. Filters with smoother transitions between passband and stopband—such as linear or Gaussian masks—help reduce these effects and produce more visually pleasing results.
* Filtering is generally less effective near the image borders due to the **periodic assumption** of the Fourier transform, which implicitly treats the image as if it repeats infinitely in all directions.

---

### ***Supplemental Materials: Code to generate distorted images and filter masks***

**<u>Feel free to skip this section if you're not interested in the implementation details.</u> 🙂**

The distorted images were generated using the code below, which you can also use to create your own variations. The distortion is produced by superimposing multiple sinusoidal gratings at different orientations. Key parameters include the *wavelength* of the gratings (in pixels) and the *angular step size*, which controls the distribution of their orientations. For this exercise, distortions were generated using wavelengths of 2.5, 5.0, and 7.5 pixels.

In [ ]:
def sinusoidal_grating_dir(shape, A, l, phi=0, angle=0, offset=0):
    """This is the sinusoidal grating function that we already have implemented
    in the previous notebook. It generates a sinusoidal grating image with the
    specified parameters.
    """
    i = np.arange(shape[1])
    j = np.arange(shape[0])
    X, Y = np.meshgrid(i, j)
    angle = np.deg2rad(angle)
    img = A * np.sin(2 * np.pi / l * ((X*np.cos(angle) + Y*np.sin(angle))) + phi) + offset
    return img.astype(np.float32)


def add_periodic_pattern(image, wavelength, step, alpha=0.6):
    # Convert to floating point and normalize
    if image.dtype == np.uint8:
        image = image.astype(np.float32)
        image /= 255

    # Add distortion: sinusoidal gratings a different orientations:
    #    angles = [0, 30, 60, 90, 120, 150]
    # Note: We only need angles from 0 to 180, since the gratings are symmetric.
    #       The gratings at 0 and 180 degrees are mirrored, and will cancel each 
    #       other out.

    # Parameters for the sinusoidal gratings
    amplitude = 1
    phase = 0
    angles = np.arange(0, 180, step)
    # Create a list of gratings
    gratings = [sinusoidal_grating_dir(shape=(image.shape), 
                                       A=amplitude, 
                                       l=wavelength, 
                                       phi=phase, 
                                       angle=angle, 
                                       offset=0) for angle in angles]
    # Sum the list of images element-wise (using numpy)
    gratings = np.sum(gratings, axis=0)
    # Normalize (floating point image, values between 0 and 1)
    gratings = (gratings - gratings.min()) / (gratings.max() - gratings.min())
    # Mix the gratings with the original image
    image = (1-alpha)*image + alpha*gratings
    return image


# Parameters for the sinusoidal gratings:
wavelength = 5    # Wavelegth of the grating (in pixels, here: 2.5, 5, 7.5)
step = 30         # Step size for the angles (in degrees)

# Load the image
image = cv.imread("../data/images/moon-walk-square.jpg", cv.IMREAD_GRAYSCALE)
image = add_periodic_pattern(image, wavelength, step)
isp.show_image(image, normalize=True)

# cv.imwrite("../data/images/moon-walk-distorted-w%g.jpg" % wavelength, 
#            image*255, [int(cv.IMWRITE_JPEG_QUALITY), 90]);

<!-- 
Development note: 

```python
# Force the input image to have a suitable size in order to avoid edge 
# effects when calculating the DFT/IDFT. See this post for details:
# https://dsp.stackexchange.com/questions/93712/
h = cv.getOptimalDFTSize(image.shape[0])
w = cv.getOptimalDFTSize(image.shape[1])
mt = (w - image.shape[1]) // 2
mb = (w - image.shape[1] - mt)
ml = (h - image.shape[0]) // 2
mr = (h - image.shape[0] - ml)
image = cv.copyMakeBorder(image, mt, mb, ml, mr, cv.BORDER_REFLECT)
cv.imwrite("../data/images/moon-walk-square-fixed.jpg", 
           image, [int(cv.IMWRITE_JPEG_QUALITY), 90]);
```
-->

Create the different masks. The mask geometries depend on the wavelength (and the angle step) of the periodic distortion created above.

In [ ]:
def circular_mask_pattern(image, wavelength, step, size):
    """Function that creates a mask with small circles along a bigger circle. 
    The circle radii are determined by the wavelength parameter."""
    mask = np.zeros_like(image)
    radius = image.shape[0]/wavelength
    angles = np.arange(0, 360, step)
    for a in angles:
        x = int(image.shape[1]/2 + radius*np.cos(np.deg2rad(a)))
        y = int(image.shape[0]/2 + radius*np.sin(np.deg2rad(a)))
        mask = cv.circle(mask, (x, y), size, 1, -1)
    mask = 1-mask
    return mask

# Ensure output folder
isp.ensure_dir("../data/images/moon-walk-masks/")

# Used for the ring masks
J, I = np.meshgrid(np.arange(image.shape[1]), np.arange(image.shape[0]))
R = np.sqrt((J - image.shape[1]//2)**2 + (I - image.shape[0]//2)**2)

# Mask 1: Low-pass filter
margin = 15
mask_low = np.zeros_like(image, dtype=np.uint8)
mask_low[R < (image.shape[0]/wavelength-margin)] = 1
cv.imwrite("../data/images/moon-walk-masks/mask1-w%g.png" % wavelength, 
           (mask_low*255).astype(np.uint8))

# Mask 2: High-pass filter
mask_high = np.zeros_like(image, dtype=np.uint8)
mask_high[R > (image.shape[0]/wavelength-margin)] = 1
cv.imwrite("../data/images/moon-walk-masks/mask2-w%g.png" % wavelength, 
           (mask_high*255).astype(np.uint8))

# Mask 3: Circular mask
size = 9
mask = circular_mask_pattern(image, wavelength, step, size=size)
mask_circ_hs = mask.copy()
cv.imwrite("../data/images/moon-walk-masks/mask3-w%g.png" % wavelength, 
           (mask_circ_hs*255).astype(np.uint8))

# Mask 4: Linearly weighted circular mask
# Apply distance transform to the circular mask
mask = circular_mask_pattern(image, wavelength, step, size=size*2)
dist = cv.distanceTransform(1-mask.astype(np.uint8), cv.DIST_L2, cv.DIST_MASK_PRECISE)
mask_circ_lin = (1-dist/np.max(dist))
cv.imwrite("../data/images/moon-walk-masks/mask4-w%g.png" % wavelength, 
           (mask_circ_lin*255).astype(np.uint8))

# Mask 5: Gaussian weighted circular mask
# Distance --> Gaussian values
dist = np.exp(-dist**2/(size**2))
mask = dist/np.max(dist)
mask_circ_gauss = (mask-mask.min())/(np.max(mask)-mask.min())
cv.imwrite("../data/images/moon-walk-masks/mask5-w%g.png" % wavelength, 
           (mask_circ_gauss*255).astype(np.uint8))

# Mask 6: Ring mask
radius = image.shape[0]/wavelength
dist = np.abs(R - radius)
mask_ring_hs = dist>size
cv.imwrite("../data/images/moon-walk-masks/mask6-w%g.png" % wavelength,
           (mask_ring_hs*255).astype(np.uint8))

# Mask 7: Linearly weighted ring mask
dist = np.clip(dist, 0, size)
mask_ring_lin = dist.astype(float) / size
cv.imwrite("../data/images/moon-walk-masks/mask7-w%g.png" % wavelength, 
           (mask_ring_lin*255).astype(np.uint8))

# Mask 8: Gaussian weighted ring mask
mask = np.exp(-dist**2/(size**2))
mask = (mask-mask.min())/(np.max(mask)-mask.min())
mask_ring_gauss = (1-mask)
cv.imwrite("../data/images/moon-walk-masks/mask8-w%g.png" % wavelength, 
           (mask_ring_gauss*255).astype(np.uint8))

# Test mask, pick one of the above masks:
#   - mask_low
#   - mask_high
#   - mask_circ_hs
#   - mask_circ_lin
#   - mask_circ_gauss
#   - mask_ring_hs
#   - mask_ring_lin
#   - mask_ring_gauss
mask = mask_ring_gauss
ft = scipy.fft.fft2(image)
ft = scipy.fft.fftshift(ft)
ft_amp = np.log(np.abs(ft))
ft_amp *= mask
isp.show_image_pair(mask, ft_amp, 
                    title1="Mask", 
                    title2="Amplitude spectrum", 
                    figsize=(9, 5),
                    normalize=True)


----

### **Fourier Transform of Non-Square Images**

<!-- Part of the solution, remove for exercise version. -->

Note: When applying the Fourier transform to non-square images, the sampling of frequencies differs along each axis, which can lead to a visually stretched or compressed appearance when the spectrum is displayed. To simplify interpretation and obtain more consistent frequency resolution in both directions, it is often convenient to convert the image to a square shape before computing the Fourier transform. This can be done by either **zero-padding** the image to a square size or **cropping** it to a square region.

The following code demonstrates this effect.

Cropping can be implemented using simple array slicing. Zero-padding, on the other hand, can be conveniently performed using the function [`cv.copyMakeBorder()`](https://docs.opencv.org/3.4/d2/de8/group__core__array.html#ga2ac1049c), which allows padding with a constant value or other strategies. Different padding modes can be selected via the `method` parameter—see the available [border types](https://docs.opencv.org/3.4/d2/de8/group__core__array.html#ga209f2f4869e304c82d07739337eae7c5) in the OpenCV documentation.


In [ ]:
# Illustrate the Fourier transform of non-square images.

# Load square and non-square version of the same image
image_nonsquare = np.zeros((400, 800))
image_nonsquare = add_periodic_pattern(image_nonsquare, 10, 15)
image_square = image_nonsquare[:min(image_nonsquare.shape), :min(image_nonsquare.shape)]

# Compute the Fourier transform
ft_square = scipy.fft.fft2(image_square)
ft_nonsquare = scipy.fft.fft2(image_nonsquare)
ft_square = scipy.fft.fftshift(ft_square)
ft_nonsquare = scipy.fft.fftshift(ft_nonsquare)
# Logarithmic scale, add plus one to avoid log(0)
ft_amp_square = np.log(np.abs(ft_square)+1)
ft_amp_nonsquare = np.log(np.abs(ft_nonsquare)+1)

isp.show_image_chain([image_square, image_nonsquare], 
                     titles=["Square", "Non-square"], 
                     figsize=(10, 5), 
                     shape=None,
                     box_aspect=0.5,
                     normalize_stretch=0.1)
isp.show_image_chain([ft_amp_square, ft_amp_nonsquare], 
                     titles=["Square", "Non-square"], 
                     figsize=(10, 5),
                     shape=None,
                     box_aspect=0.5,
                     normalize_stretch=0.1)

In [ ]:
def make_image_square(image, method=cv.BORDER_CONSTANT):
    """Use zero-padding to make the image square."""
    if image.shape[0] == image.shape[1]:
        return image
    max_dim = max(image.shape)
    top = bottom = left = right = 0
    if image.shape[0] < max_dim:
        top = bottom = (max_dim - image.shape[0]) // 2
    if image.shape[1] < max_dim:
        left = right = (max_dim - image.shape[1]) // 2
    return cv.copyMakeBorder(image, top, bottom, left, right, method, value=image.mean())

In [ ]:
# Code to illustrate the effect of zero-padding on the Fourier transform
# See also exercise 5 (Border handling).

image_nonsquare = np.zeros((400, 800))
image_nonsquare = add_periodic_pattern(image_nonsquare, 10, 15)

isp.show_header("Zero-padding to make the image square")
isp.show_image_grid([image_nonsquare, None], titles=["Non-square image", None],
               suppress_info=True)

# Read in the OpenCV docs about the different border methods.
images_square = []
method_label_map = {
    cv.BORDER_CONSTANT: "Constant",
    cv.BORDER_REPLICATE: "Replicate",
    cv.BORDER_REFLECT: "Reflect",
    #cv.BORDER_REFLECT101: "Reflect101",
}

for method, label in method_label_map.items():
    # Make the image square
    image_square = make_image_square(image_nonsquare, method=method)
    
    # Compute the Fourier transform
    ft_square = scipy.fft.fft2(image_square)
    ft_square = scipy.fft.fftshift(ft_square)
    # Logarithmic scale, add plus one to avoid log(0)
    ft_amp_square = np.log(np.abs(ft_square)+1)
    
    # Display the result
    isp.show_image_grid([image_square, ft_amp_square], 
                         titles=["Square – Border mode: %s" % label, "Amplitude spectrum"], 
                         figsize=(9, 5), 
                         suppress_info=True,
                         )

---

<a id='exercise2'></a>

## **☆ Exercise 2: Interpretation of spectra**

Take a look at the amplitude spectrum of the image below, which displays a honeycomb pattern.

![Honeycomb](../data/images/honeycomb1.jpg)


### **Instructions**

Recall that the Fourier transform is generally complex-valued. Each element of the transform (an NxM array) is a complex number that can be uniquely represented by its magnitude $|z|$ and phase $\varphi = \arg(z)$, such that  
$\quad z = |z| e^{i\varphi}$

1. Load the image *honeycomb1.jpg* (or *honeycomb2.jpg*) in grayscale.  
2. Compute and visualize the **magnitude (amplitude)** and **phase** spectra.  
3. Answer: What information can you extract from the magnitude and phase spectra of the image?  
4. Reconstruct the image from its Fourier transform by:  
   * setting the magnitude to 1 (i.e., keeping only the phase information),  
   * setting the phase to 0 (i.e., keeping only the magnitude and discarding phase information).  
5. Reflect: What can you infer from these reconstructions?  



In [ ]:
######################
###    EXERCISE    ###
######################

# 1) Load the honeycomb image "honeycomb1.jpg" (or "honeycomb2.jpg") 
image = ...

# 2) Display the image and its Fourier transform (magnitude and phase)
ft = ...
ft_amp = ...
ft_phase = ...

# Display the spectra using our convenience function isp.show_image()
# (The argument "normalize stretch" is used to enhance the spectrum of the 
# spectra to see more structure). Alternatively, just use normalize=True
isp.show_image_chain([image, ft_amp, ft_phase], normalize_stretch=0.1);

# 3) Question: What can be seen in the Fourier transform?
...

# 4) Reconstruct the image from the Fourier transform and display it
#    Here: Compute the inverse Fourier transform three times:
#    - Standard inverse Fourier transform
#    - Inverse Fourier transform with the phase set to zero
#    - Inverse Fourier transform with the amplitude set to one

# 5) Observations?
...

In [ ]:
######################
###    SOLUTION    ###
######################

# Load the honeycomb image
image = cv.imread("../data/images/honeycomb3.jpg", cv.IMREAD_GRAYSCALE)
image = cv.imread("../data/images/honeycomb2.jpg", cv.IMREAD_GRAYSCALE)
image = cv.imread("../data/images/honeycomb1.jpg", cv.IMREAD_GRAYSCALE)

# You can use any other image as well...
# image = cv.imread("../data/images/moon-walk.jpg", cv.IMREAD_GRAYSCALE)

# Make the image square (see comment above about non-square images)
if True:
    image = image[:min(image.shape), :min(image.shape)]

# Compute the Fourier transform
ft = scipy.fft.fft2(image)
ft = scipy.fft.fftshift(ft)
# Compute the amplitude spectrum. Use the log to improve the interpretabilty
# of the amplitude spectrum. We add 1 to avoid log(0) = -inf
magnitude = np.log(np.abs(ft)+1) 
# Compute the phase spectrum
phase = np.angle(ft)

# Display the spectra using our convenience function isp.show_image()
# (The argument "normalize stretch" is used to enhance the spectrum of the 
# spectra to see more structure). Alternatively, just use normalize=True
isp.show_image_chain([image, magnitude, phase], 
                     titles=["Input image", "Amplitude spectrum", "Phase spectrum"], 
                     normalize_stretch=0.1);

# Compute the inverse Fourier transform three times:
# - Standard inverse Fourier transform
# - Inverse Fourier transform with the phase set to zero
# - Inverse Fourier transform with the amplitude set to one

# Phase = 0: 
#  --> ft_amp = r*exp(i*0) = r 
#  --> apply inverse FT
ft_amp = np.abs(ft)
ft_amp = scipy.fft.ifftshift(ft_amp)
# The inverse Fourier transform should be real valued, but due
# to numerical errors, we need to take the absolute value.
image_rec_amp = np.abs(scipy.fft.ifft2(ft_amp))

# Amplitude = 1:
#  --> ft_phase = 1*exp(i*phi) = exp(i*phi)
#  --> apply inverse FT
ft_phase = np.exp((0+1j)*phase)
ft_phase = scipy.fft.ifftshift(ft_phase)
# The inverse Fourier transform should be real valued, but due
# to numerical errors, we need to take the absolute value.
image_phase_rec = np.abs(scipy.fft.ifft2(ft_phase))

# Visualize
isp.show_image_chain([image, image_rec_amp, image_phase_rec], 
                     titles=["Input image", 
                             "Reconstruction with phase=0", 
                             "Reconstruction with amplitude=1"],
                     normalize_stretch=0.1)

### **Observations**

- The honeycomb pattern is clearly reflected in the amplitude spectrum: three dominant axes appear, corresponding to the main directions of the hexagonal structure. In contrast, the phase spectrum does not exhibit an immediately obvious structure.
- The amplitude spectrum describes the distribution of energy across frequency components. However, on its own, it is not sufficient for image reconstruction: without phase information, the overall shape and spatial structure are lost.
- The phase spectrum contains the essential information about the spatial arrangement of structures in the image. Even with uniform amplitude, the main objects and patterns remain clearly recognizable.

---

<a id='exercise3'></a>

## **☆ Exercise 3: Spatial filtering with kernels**

In this exercise, you will explore how different filter kernels affect an image. Feel free to experiment by designing your own kernels and observing their effects.

**Note:** Kernels must have an odd number of rows and columns so that their center aligns with a specific pixel.

### **Instructions**

* Explore the following resources for examples of commonly used kernels:  
  * [Wikipedia: Image processing kernels](https://en.wikipedia.org/wiki/Kernel_(image_processing))  
  * [setosa.io: Interactive kernel visualizations](https://setosa.io/ev/image-kernels/)  
* Select a few kernels and implement them in Python.  
* Apply the kernels to the image *"camera.png"* using one of the following methods:  
  * OpenCV: [cv.filter2D()](https://docs.opencv.org/3.4/d4/dbd/tutorial_filter_2d.html)  
  * SciPy: [scipy.signal.convolve2d()](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.convolve2d.html)  

Remember that applying a filter corresponds to convolving an image with a kernel. As discussed in the lecture, convolution can also be performed in the Fourier domain, which is often more efficient for large kernels. For small kernels (typically smaller than 11×11), however, direct convolution is usually faster. Functions such as `cv.filter2D()` and `convolve2d()` handle this efficiently internally.



In [ ]:
######################
###    EXERCISE    ###
######################

image = cv.imread("../data/images/camera.png", cv.IMREAD_GRAYSCALE)

kernel = ...
image_filt = ... 

isp.show_image_pair(image, image_filt, 
                    title1="Original", 
                    title2="Filtered", 
                    figsize=(9, 5));

In [ ]:
######################
###    SOLUTION    ###
######################

image = cv.imread("../data/images/camera.jpg", cv.IMREAD_GRAYSCALE)

# # Everything works also for color images!
# image = cv.imread("../data/images/flowers.jpg", cv.IMREAD_COLOR)
# image = cv.cvtColor(image, cv.COLOR_BGR2RGB)

# We collect the different kernels as (name + kernel) pairs in a dictionary.
# We then will use this dictionary to apply the kernels to the image and
# display the results in a grid.
kernels = {}

identity = np.array([[0, 0, 0],
                     [0, 1, 0],
                     [0, 0, 0]])
kernels["Identity"] = identity

# By adding None, we just add an empty image to the grid
# (This is just for the organization of the grid)
kernels["Identity spacer 1"] = None
kernels["Identity spacer 2"] = None

###########################################
# Blurring
###########################################

# Box blur filter (3x3)
#box3x3 = np.ones((3, 3), np.float32) / 9
box_3x3 = np.array([[ 1, 1, 1],
                    [ 1, 1, 1],
                    [ 1, 1, 1]]) / 9
kernels["Box filter 3x3"] = box_3x3

# Box blur filter (5x5)
box_5x5 = np.ones((5, 5), np.float32) / 25
kernels["Box filter 5x5"] = box_5x5

# Box blur filter (nxn)
n = 11
box_nxn = np.ones((n, n), np.float32) / (n*n)
kernels["Box filter nxn"] = box_nxn

# Gaussian blur filter (3x3)
gaussian_3x3 = np.array([[1, 2, 1],
                         [2, 4, 2],
                         [1, 2, 1]]) / 16
kernels["Gaussian filter 3x3"] = gaussian_3x3

# Gaussian blur filter (5x5)
gaussian_5x5 = np.array([[1, 4, 6, 4, 1],
                         [4, 16, 24, 16, 4],
                         [6, 24, 36, 24, 6],
                         [4, 16, 24, 16, 4],
                         [1, 4, 6, 4, 1]]) / 256
kernels["Gaussian filter 5x5"] = gaussian_5x5

# Gaussian blur filter (nxn)
n = 31
gaussian_nxn = cv.getGaussianKernel(n, -1) @ cv.getGaussianKernel(n, -1).T
kernels["Gaussian filter nxn"] = gaussian_nxn

###########################################
# Edge / ridge detection filters
###########################################

# Prewitt filter to detect horizontal edges
prewitt_x = np.array([[-1, 0, 1],
                      [-1, 0, 1],
                      [-1, 0, 1]])
kernels["Prewitt right"] = prewitt_x

prewitt_x = np.array([[ 1, 0, -1],
                      [ 1, 0, -1],
                      [ 1, 0, -1]])
kernels["Prewitt left"] = prewitt_x
kernels["Prewitt spacer 1"] = None

# Prewitt filter to detect vertical edges
prewitt_y = np.array([[-1, -1, -1],
                      [0, 0, 0],
                      [1, 1, 1]])
kernels["Prewitt bottom"] = prewitt_y

prewitt_y = np.array([[1, 1, 1],
                      [0, 0, 0],
                      [-1, -1, -1]])
kernels["Prewitt top"] = prewitt_y
kernels["Prewitt spacer 2"] = None

# Sobel filter to detect horizontal edges
sobel_x = np.array([[-1,  0,  1],
                    [-2,  0,  2],
                    [-1,  0,  1]])
kernels["Sobel right"] = sobel_x

sobel_x = np.array([[1,  0,  -1],
                    [2,  0,  -2],
                    [1,  0,  -1]])
kernels["Sobel left"] = sobel_x

# Sobel filter to detect vertical edges
sobel_y = np.array([[-1, -2, -1],
                    [ 0,  0,  0],
                    [ 1,  2,  1]])
kernels["Sobel bottom"] = sobel_y

# Laplacian filter (3x3) to detect edges 
laplacian_3x3 = np.array([[ 0,  1,  0],
                          [ 1, -4,  1],
                          [ 0,  1,  0]])
kernels["Laplacian 3x3"] = laplacian_3x3

# Laplacian filter (3x3) to detect edges, extended version that 
# increases sensitivity to diagonal edges
laplacian_3x3diag = np.array([[ 1,  1,  1],
                              [ 1, -8,  1],
                              [ 1,  1,  1]])
kernels["Laplacian 3x3_diag: outline"] = laplacian_3x3diag

# Laplacian filter (5x5) to detect edges in all directions
laplacian_5x5 = np.array([[ 0,  0,   1,  0,  0],
                          [ 0,  1,   2,  1,  0],
                          [ 1,  2, -16,  2,  1],
                          [ 0,  1,   2,  1,  0],
                          [ 0,  0,   1,  0,  0]])
kernels["Laplacian 5x5"] = laplacian_5x5

###########################################
# Sharpening filters
###########################################

# Sharpening filter (3x3)
sharpen_3x3 = np.array([[ 0, -1,  0],
                        [-1,  5, -1],
                        [ 0, -1,  0]])
kernels["Sharpen 3x3"] = sharpen_3x3

# Sharpening filter (5x5)
sharpen_5x5 = np.array([[ 0,  0, -1,  0,  0],
                        [ 0, -1, -2, -1,  0],
                        [-1, -2, 17, -2, -1],
                        [ 0, -1, -2, -1,  0],
                        [ 0,  0, -1,  0,  0]])
kernels["Sharpen 5x5"] = sharpen_5x5

# Unsharp masking filter (3x3)
alpha = 8
unsharp_masking = (np.array([[ 0,  0,  0],
                             [ 0,  1,  0],
                             [ 0,  0,  0]]) +
                   alpha/16 * np.array([[-1, -2, -1],
                                       [-2, 12, -2],
                                       [-1, -2, -1]]))
kernels["Unsharp masking 3x3"] = unsharp_masking

###########################################
# Embossing filters
###########################################

# Embossing filter (3x3)
emboss_3x3 = np.array([[-2, -1,  0],
                       [-1,  1,  1],
                       [ 0,  1,  2]])
kernels["Emboss 3x3"] = emboss_3x3

# Embossing filter (5x5)
emboss_5x5 = np.array([[-2,  0, -1,  0,  0],
                       [ 0, -2, -1,  0,  0],
                       [-1, -1,  1,  1,  1],
                       [ 0,  0,  1,  2,  0],
                       [ 0,  0,  1,  0,  2]])
kernels["Emboss 5x5"] = emboss_5x5
kernels["emboss_spacer"] = None

####################################################################################
# Apply the kernels to the image one by one and display the results
results = {}
for name, kernel in kernels.items():
    # Required for the "spacer" images
    if kernel is None:
        results[name] = None
        continue

    # Finally! Here's the filtering step:
    if True:
        # Equivalent call using OpenCV
        # IMPORTANT: We need to specify the data type of the output
        #            We have two main options here: cv.CV_32F or cv.CV_8U
        #            Note that cv.CV_8U will clip the values to the range [0, 255]
        #            This is not always desired, especially for edge detection!!!
        #            However, other filters (e.g., embossing) might look better
        #            with clipping.
        dtype_out = cv.CV_32F
        dtype_out = cv.CV_8U
        image_filtered = cv.filter2D(image, dtype_out, kernel)
    else:
        # Equivalent call using scipy
        # This method uses floating point numbers 
        image_filtered = scipy.signal.convolve2d(image, 
                                                 kernel, 
                                                 mode="same", 
                                                 boundary="symm")

    #image_filtered = np.abs(image_filtered)
    #image_filtered = np.clip(image_filtered, 0, 255)

    # Be aware of the data types!
    print("Filter:", name)
    print("  dtype input image:", image.dtype)
    print("  dtype output image:", image_filtered.dtype)
    print("  dtype kernel:", kernel.dtype)
    print("  value range: (%.1f, %.1f)" % (image_filtered.min(), image_filtered.max()))
    print()

    results[name] = image_filtered


# Plot the results in a grid
nrows = round(len(results)/3+1)
isp.show_image_grid(results.values(), 
                    titles=results.keys(), 
                    ncols=3, 
                    figsize=(9, nrows*3), 
                    #normalize=(0,1),
                    suppress_info=True)

---

<a id='exercise4'></a>

## **☆ Exercise 4: Edge detection**

In the previous exercise, you saw that certain kernels can be used to enhance edges in an image by approximating spatial derivatives. Common examples include the Sobel and Prewitt filters. Another widely used operator is the Laplacian filter.

Note: Computing derivatives is inherently sensitive to noise, as differentiation amplifies high-frequency components (where noise is typically present). To mitigate this effect, it is common practice to smooth the image before computing its derivative.

Edge detection can be viewed as a two-step process:
1. **Enhance edges** by applying a transformation (e.g., gradient-based or Laplacian filtering).  
2. **Detect edges** by applying a threshold to obtain a binary edge mask.

In this exercise, you will implement such an edge detection pipeline.

### **Instructions**

1. Load the image in grayscale.  
2. Smooth the image using cv.GaussianBlur().  
3. Choose and apply a suitable edge-enhancing kernel.  
4. Apply thresholding to generate a binary mask that highlights the edges.  

In [ ]:
######################
###    EXERCISE    ###
######################

# 1) Load the image "camera.jpg" and display it.
image = ...

# 2) Smooth the image using cv.GaussianBlur()
image_smoothed = ...

# 3) + 4) Implement your edge detection method
...
edges = ...

# Visualization:
isp.show_image_pair(image, edges, 
                    title1="Original", 
                    title2="Edges", 
                    figsize=(9, 5))

In [ ]:
######################
###    SOLUTION    ###
######################

# Load image as grayscale 
image = cv.imread("../data/images/camera.jpg", cv.IMREAD_GRAYSCALE)

# Smooth the input image using a Gaussian filter
# This will help to reduce noise and improve the edge detection
image_smoothed = cv.GaussianBlur(image, (5, 5), 0)


################################################################################
# METHOD 1: Sobel filter
################################################################################
# Use the Sobel filter to detect horizontal and vertical edges
sobel_x = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]])
sobel_y = np.array([[-1, -2, -1],
                    [ 0,  0,  0],
                    [ 1,  2,  1]])

# With these kernels, we can compute the gradient vector: (dx, dy)
dx = cv.filter2D(image_smoothed, cv.CV_32F, sobel_x)
dy = cv.filter2D(image_smoothed, cv.CV_32F, sobel_y)

# Compute the gradient magnitude
gradient = np.sqrt(dx**2 + dy**2)

# Threshold the gradient magnitude
threshold = 200
edges = (gradient > threshold)

# Display the results
isp.show_image_chain([image, gradient, edges], 
                      titles=["Input image", "Gradient magnitude: Sobel", "Edges: Sobel"],
                      figsize=(9, 5))
#isp.save_figure(path="edge-detection-sobel.png", transparent=True)

################################################################################
# METHOD 2: Use the Laplacian filter
################################################################################

# Laplacian filter to detect edges
laplacian = np.array([[ 0,  1,  0],
                      [ 1, -4,  1],
                      [ 0,  1,  0]])
# Choose threshold
threshold = 15


laplacian = np.array([[ 1,  4,  1],
                      [ 4, -20, 4],
                      [ 1,  4,  1]])
threshold = 75

# Apply the Laplacian filter
image_filtered = cv.filter2D(image_smoothed, cv.CV_32F, laplacian)

# Threshold the result
edges = np.abs(image_filtered) > threshold

# Well... Thresholding is not correct here. For the Laplacian filter, we want to 
# detect zero-crossings. In fact, image_filtered consists of positive and 
# negative values. We can detect edges where the sign changes. To detect the 
# sign change, we can use the following approach:
# 1) Threshold the image to get positive and negative values.
pos = image_filtered > threshold
neg = image_filtered < -threshold
# 2) Pad the image (with zeros), we need this to avoid border effects
pos = np.pad(pos, ((1, 1), (1, 1)), mode="constant")
neg = np.pad(neg, ((1, 1), (1, 1)), mode="constant")
# 3) Detect sign changes in horizontal, vertical or diagonal directions
horizontal = (pos[2:, 1:-1] & neg[:-2, 1:-1]) | (neg[2:, 1:-1] & pos[:-2, 1:-1])
vertical = (pos[1:-1, 2:] & neg[1:-1, :-2]) | (neg[1:-1, 2:] & pos[1:-1, :-2])
diagonal1 = (pos[2:, 2:] & neg[:-2, :-2]) | (neg[2:, 2:] & pos[:-2, :-2])
diagonal2 = (pos[2:, :-2] & neg[:-2, 2:]) | (neg[2:, :-2] & pos[:-2, 2:])
# 4) A pixel is an edge pixel if it sees a sign change in any direction
edges = horizontal | vertical | diagonal1 | diagonal2

# Display the results
isp.show_image_chain([image, image_filtered, edges], 
                      titles=["Input image", "Laplacian filter", "Edges: Laplacian"],
                      figsize=(9, 5))
#isp.save_figure(path="edge-detection-laplacian.png", transparent=True)

Note that the edges detected using the Laplacian filter tend to appear thinner than those detected with the Sobel filter. This is because the Laplacian detects zero-crossings, which are typically more localized than the gradient magnitude computed by Sobel.

Overall, the result still leaves room for improvement and is highly sensitive to the choice of threshold. For better performance, more advanced methods such as the [**Canny edge detector**](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html) can be used. Canny edge detection includes several techniques to suppress noise and reduce false detections: Non-maximum suppression, double thresholding, and edge tracking by hysteresis.

The results are typically much more robust and visually accurate:

In [ ]:
################################################################################
# METHOD 3: Canny edge detection
################################################################################

# Double thresholding
thr_lower = 100
thr_upper = 200

# Compare results with and without smoothing
edges_no_smoothing = cv.Canny(image, thr_lower, thr_upper)
edges_after_smoothing = cv.Canny(image_smoothed, thr_lower, thr_upper)

isp.show_image_chain([image, edges_no_smoothing, edges_after_smoothing], 
                      titles=["Input image", "Edges (Canny)", "Edges: Canny, smoothed"],
                      figsize=(9, 5))

In [ ]:
# Extra: Illustrate how the gradient estimates change with/without smoothing.
dx = cv.filter2D(image, cv.CV_32F, sobel_x)
dy = cv.filter2D(image, cv.CV_32F, sobel_y)
gradient_nonsmoothed = np.sqrt(dx**2 + dy**2)
dx = cv.filter2D(image_smoothed, cv.CV_32F, sobel_x)
dy = cv.filter2D(image_smoothed, cv.CV_32F, sobel_y)
gradient_smoothed = np.sqrt(dx**2 + dy**2)

isp.show_image_chain([image, gradient_nonsmoothed, edges_no_smoothing.astype(bool)], 
                      titles=["Input image", "Gradient magnitude", "Edges: Canny"],
                      figsize=(9, 5));
#isp.save_figure(path="edge-detection-canny.png", transparent=True)

isp.show_image_chain([image_smoothed, gradient_smoothed, edges_after_smoothing.astype(bool)], 
                      titles=["Input image smoothed", "Gradient magnitude", "Edges: Canny"],
                      figsize=(9, 5))
#isp.save_figure(path="edge-detection-canny-smoothed.png", transparent=True)

---

<a id='exercise5'></a>

## **☆ Exercise 5: Border handling**

In both spectral filtering (e.g., in Exercise 1) and spatial filtering (e.g., convolution with a kernel), proper handling of image borders is essential to avoid unwanted artifacts. This section illustrates how OpenCV provides convenient ways to manage image boundaries.

In spectral filtering, artifacts often arise from discontinuities at the image borders, since the Discrete Fourier Transform (DFT) implicitly assumes the image is periodic. One effective way to reduce such artifacts is to extend the image using mirrored or repeated borders before processing.

OpenCV functions such as [`cv.filter2D()`](https://docs.opencv.org/4.x/d4/d86/group__imgproc__filter.html#ga27c049795ce870216ddfb366086b5a04) support an optional `borderType` parameter, which defines how borders are handled. The default setting, `cv.BORDER_DEFAULT`, corresponds to a reflection (mirroring) of the border pixels. A full list of available options can be found [here](https://docs.opencv.org/4.x/d2/de8/group__core__array.html#ga209f2f4869e304c82d07739337eae7c5).

### **Instructions**

* Load an arbitrary image  
* Illustrate the effect of different border types  

In [ ]:
######################
###    EXERCISE    ###
######################
...

In [ ]:
######################
###    SOLUTION    ###
######################

def mirror_image(image, 
                 border_width, 
                 border_type=cv.BORDER_REFLECT_101,
                 value=None):
    """Add a border of a certain width to the image"""
    # Create 3 times bigger image, with mirrored borders
    top = bottom = left = right = border_width
    # Argument 'value' is only used for BORDER_CONSTANT
    image = cv.copyMakeBorder(image, 
                              top=top, 
                              bottom=bottom,
                              left=left,
                              right=right,
                              borderType=border_type,
                              value=value)
    return image
    
def demirror_image(image, border_width):
    """Renove a border previously added by mirror_image()"""
    # Crop the image to its original size
    image = image[border_width:-border_width, 
                  border_width:-border_width]
    return image

image = cv.imread("../data/images/flowers.jpg", cv.IMREAD_COLOR)
image = cv.imread("../data/images/veggies.jpg", cv.IMREAD_COLOR)
image = cv.cvtColor(image, cv.COLOR_BGR2RGB)

border_width = 100
border_types = {}

# BORDER_REFLECT and BORDER_REFLECT_101 are almost the same:
#   BORDER_REFLECT: fedcba|abcdefgh|hgfedcb
#   BORDER_REFLECT_101: gfedcb|abcdefgh|gfedcba
border_types["BORDER_CONSTANT"] = cv.BORDER_CONSTANT
border_types["cv.BORDER_REFLECT"] = cv.BORDER_REFLECT 
border_types["cv.BORDER_REFLECT_101"] = cv.BORDER_REFLECT_101
border_types["cv.BORDER_WRAP"] = cv.BORDER_WRAP
border_types["cv.BORDER_REPLICATE"] = cv.BORDER_REPLICATE

results = {}
for name, border_type in border_types.items():
    # Mean color in the image
    mean_color = np.mean(image, axis=(0, 1))
    image_border = mirror_image(image, border_width, border_type, value=mean_color)
    results[name] = image_border

isp.show_image_grid(results, ncols=3, figsize=(9, 6), suppress_info=True)